# 00_preprocessing — Preprocessing, registration and response estimation

**Manuscript:** Methods 'MRI acquisition and preprocessing', 'ROI definition and response estimation'; Supplementary S2 (pipelines), S3 (ROI coverage).

Functional runs cover occipital cortex only, so registration used header-initialised mutual information (`run_method3_header_mi_*.sbatch`). The primary pipeline resamples every volume once with a run-constant transform; the head-motion-corrected pipeline composes each volume's MCFLIRT matrix with that transform before the same single resampling (`hmc_reanalysis/`). `run_full_dataset_C010.py` fits the per-run GLM and writes the run x hue x voxel amplitude arrays (dataset token `C010`) that every later stage reads. The notebook loads the committed QC summary (framewise displacement, ROI coverage, ROI tSNR under both pipelines).

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_00_preprocessing.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/preprocessing_qc_summary.json` | `scripts/motion_qc_summary.py, scripts/analyze_registration_quality.py, scripts/hmc_v2/analyze_hmc.sh` | framewise displacement, ROI coverage, ROI tSNR under both pipelines |
| `results/bbr_vs_mi_displacement.json` | `(fMRIPrep BBR vs header-initialised MI, server QC)` | slab centroid displacement between the two registration routes |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

q = J("preprocessing_qc_summary.json")
fd = q["framewise_displacement_mm"]; cov = q["roi_coverage"]; tsnr = q["roi_tsnr"]

V.start("00_preprocessing")

### Head motion (Supplementary S2, 'The two pipelines')
Framewise displacement (Power et al. 2012, rotations on a 50 mm sphere), nine analysed participants.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 00.01 | S2 ¶1 | mean FD, nine analysed participants (mm) | `0.318` |
| 00.02 | S2 ¶1 | SD of FD, nine participants | `0.044` |
| 00.03 | S2 ¶1 | mean FD, controls | `0.313` |
| 00.04 | S2 ¶1 | SD of FD, controls | `0.042` |
| 00.05 | S2 ¶1 | mean FD, CVD | `0.338` |
| 00.06 | S2 ¶1 | SD of FD, CVD | `0.046` |
| 00.07 | S2 ¶1 | percentage of volumes with FD > 0.5 mm | `16.2` |

In [2]:
g = fd["exp1_groups"]
print({k: (round(v["mean_fd_mm"], 3), round(v["sd"], 3)) for k, v in g.items()})
V.check('00.01', 'S2 ¶1 | mean FD, nine analysed participants (mm)', g["all_analysed"]["mean_fd_mm"], 0.318, nd=3)
V.check('00.02', 'S2 ¶1 | SD of FD, nine participants', g["all_analysed"]["sd"], 0.044, nd=3)
V.check('00.03', 'S2 ¶1 | mean FD, controls', g["HC"]["mean_fd_mm"], 0.313, nd=3)
V.check('00.04', 'S2 ¶1 | SD of FD, controls', g["HC"]["sd"], 0.042, nd=3)
V.check('00.05', 'S2 ¶1 | mean FD, CVD', g["CVD"]["mean_fd_mm"], 0.338, nd=3)
V.check('00.06', 'S2 ¶1 | SD of FD, CVD', g["CVD"]["sd"], 0.046, nd=3)
V.check('00.07', 'S2 ¶1 | percentage of volumes with FD > 0.5 mm', fd["exp1_pct_volumes_fd_gt_0p5"], 16.2, nd=1)

{'all_analysed': (0.318, 0.044), 'HC': (0.313, 0.042), 'CVD': (0.338, 0.046), 'excluded_sub10': (0.321, 0.0)}
[OK ] 00.01 S2 ¶1 | mean FD, nine analysed participants (mm): produced=0.3184  reported=0.318
[OK ] 00.02 S2 ¶1 | SD of FD, nine participants: produced=0.04422  reported=0.044
[OK ] 00.03 S2 ¶1 | mean FD, controls: produced=0.3128  reported=0.313
[OK ] 00.04 S2 ¶1 | SD of FD, controls: produced=0.04214  reported=0.042
[OK ] 00.05 S2 ¶1 | mean FD, CVD: produced=0.3379  reported=0.338
[OK ] 00.06 S2 ¶1 | SD of FD, CVD: produced=0.04576  reported=0.046
[OK ] 00.07 S2 ¶1 | percentage of volumes with FD > 0.5 mm: produced=16.24  reported=16.2


### tSNR cost of head-motion correction (Supplementary S2)
ROI temporal SNR under the primary and the head-motion-corrected pipeline; the manuscript reports the range of the reduction over the four ROIs.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 00.08 | S2 ¶1 | smallest tSNR reduction over ROIs (%) | `1.7` |
| 00.09 | S2 ¶1 | largest tSNR reduction over ROIs (%) | `2.7` |

In [3]:
red = {roi: -v["pct_change"] for roi, v in tsnr.items()}
print({k: round(v, 2) for k, v in red.items()})
V.check('00.08', 'S2 ¶1 | smallest tSNR reduction over ROIs (%)', min(red.values()), 1.7, nd=1)
V.check('00.09', 'S2 ¶1 | largest tSNR reduction over ROIs (%)', max(red.values()), 2.7, nd=1)

{'V1': 2.69, 'V2': 1.88, 'V3': 1.66, 'hV4': 1.75}
[OK ] 00.08 S2 ¶1 | smallest tSNR reduction over ROIs (%): produced=1.662  reported=1.7
[OK ] 00.09 S2 ¶1 | largest tSNR reduction over ROIs (%): produced=2.695  reported=2.7


### ROI coverage (Supplementary S3)
Intersection of the Wang-atlas ROI with each participant's BOLD brain mask in MNI space, mean over four ROIs and six runs.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 00.10 | S3 | mean ROI coverage (%) | `83.5` |
| 00.11 | S3 | SD of ROI coverage (%) | `22.7` |
| 00.12 | S3 | ROI voxels with reliable stimulus-evoked responses (%) | `99.5` |
| 00.13 | S3 | lowest coverage in the sample (%) | `30.8` |
| 00.14 | S3 | number of analysed participants | `9` |

In [4]:
print({s: round(v["mean_roi_coverage_ratio"] * 100, 1) for s, v in cov["per_subject"].items()})
V.check('00.10', 'S3 | mean ROI coverage (%)', cov["mean_pct"], 83.5, nd=1)
V.check('00.11', 'S3 | SD of ROI coverage (%)', cov["sd_pct"], 22.7, nd=1)
V.check('00.12', 'S3 | ROI voxels with reliable stimulus-evoked responses (%)', cov["glm_valid_voxel_pct"], 99.5, nd=1)
V.check('00.13', 'S3 | lowest coverage in the sample (%)', cov["min_pct"], 30.8, nd=1)
V.check('00.14', 'S3 | number of analysed participants', len(cov["analysed_participants"]), 9, mode='eq')

{'sub-01': 77.9, 'sub-02': 69.2, 'sub-03': 99.0, 'sub-04': 99.5, 'sub-05': 100.0, 'sub-07': 30.8, 'sub-06': 100.0, 'sub-08': 84.2, 'sub-09': 90.7, 'sub-10': 100.0}
[OK ] 00.10 S3 | mean ROI coverage (%): produced=83.47  reported=83.5
[OK ] 00.11 S3 | SD of ROI coverage (%): produced=22.65  reported=22.7
[OK ] 00.12 S3 | ROI voxels with reliable stimulus-evoked responses (%): produced=99.54  reported=99.5
[OK ] 00.13 S3 | lowest coverage in the sample (%): produced=30.8  reported=30.8
[OK ] 00.14 S3 | number of analysed participants: produced=9  reported=9


### Quantities without a committed artifact (Supplementary S2)
These were measured on the server from the field maps and the registration transforms; they are reported here as pointers only.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 00.15 | S2 'Susceptibility distortion' | field-map displacement within ROIs 0.01-0.76 voxels (0.02-1.52 mm); within-ROI variation 0.05-0.21 voxels (0.38 in one participant) | `no committed artifact (server field-map computation)` |
| 00.16 | S2 'Registration' | run-to-run displacement of the MI solution 0.9-4.2 mm; defacing moved it by 1.9 mm (deutan) and 9.4 mm (protan) | `no committed artifact (server registration logs)` |
| 00.17 | S2 'Registration' | BBR snapped the slab about 10 mm off on visual inspection; MI within about 1 mm | `visual criterion; bbr_vs_mi_displacement.json gives centroid displacement for three participants (descriptive)` |

In [5]:
b = J("bbr_vs_mi_displacement.json")
print("BBR vs MI slab-centroid displacement (mm):", {s: v["displacement_mm"] for s, v in b["subjects"].items()})
V.flag('00.15', "S2 'Susceptibility distortion' | field-map displacement within ROIs 0.01-0.76 voxels (0.02-1.52 mm); within-ROI variation 0.05-0.21 voxels (0.38 in one participant)", 'no committed artifact (server field-map computation)', 'no committed artifact (server field-map computation)')
V.flag('00.16', "S2 'Registration' | run-to-run displacement of the MI solution 0.9-4.2 mm; defacing moved it by 1.9 mm (deutan) and 9.4 mm (protan)", 'no committed artifact (server registration logs)', 'no committed artifact (server registration logs)')
V.flag('00.17', "S2 'Registration' | BBR snapped the slab about 10 mm off on visual inspection; MI within about 1 mm", 'visual criterion; bbr_vs_mi_displacement.json gives centroid displacement for three participants (descriptive)', 'visual criterion; bbr_vs_mi_displacement.json gives centroid displacement for three participants (descriptive)')

BBR vs MI slab-centroid displacement (mm): {'sub-01': 3.37, 'sub-03': 2.49, 'sub-06': 5.12}
[-- ] 00.15 S2 'Susceptibility distortion' | field-map displacement within ROIs 0.01-0.76 voxels (0.02-1.52 mm); within-ROI variation 0.05-0.21 voxels (0.38 in one participant): reported=no committed artifact (server field-map computation)  NO COMMITTED ARTIFACT: no committed artifact (server field-map computation)
[-- ] 00.16 S2 'Registration' | run-to-run displacement of the MI solution 0.9-4.2 mm; defacing moved it by 1.9 mm (deutan) and 9.4 mm (protan): reported=no committed artifact (server registration logs)  NO COMMITTED ARTIFACT: no committed artifact (server registration logs)
[-- ] 00.17 S2 'Registration' | BBR snapped the slab about 10 mm off on visual inspection; MI within about 1 mm: reported=visual criterion; bbr_vs_mi_displacement.json gives centroid displacement for three participants (descriptive)  NO COMMITTED ARTIFACT: visual criterion; bbr_vs_mi_displacement.json gives centro

In [6]:
V.summary()


=== 00_preprocessing: 14/14 numeric checks reproduced exactly; 0 within one unit of the last printed digit; 0 mismatch, 0 error, 3 pointer-only ===
